# mIF Pipeline Python API Stages (Crop)

This notebook demonstrates the intended stage-by-stage Python API flow after the artifact-first refactor.

Expected stage model:
- `merge`: write `full_merge.ome.tif`
- `instanseg`: run in an InstanSeg-compatible env against `full_merge.ome.tif` using `instanseg.channels`
- `nimbus`: optional, produces a table artifact
- `assemble_spatialdata`: run in a modern Harpy + SpatialData env using `full_merge.ome.tif`, label TIFFs, and optional Nimbus CSV

This notebook does not try to bridge environments automatically. It just shows the correct Python API calls and expected artifacts.


In [ ]:
from pathlib import Path
import json
import platform
import traceback

import mif_pipeline
from mif_pipeline import (
    assemble_spatialdata,
    load_config,
    merge_slide_ometiffs,
    run_nimbus_chunked,
    setup_slide,
)
from mif_pipeline.config import get_slide_config
import mif_pipeline.spatialdata_builder as spatialdata_builder_module

print("python:", platform.python_version())
print("mif_pipeline:", getattr(mif_pipeline, "__file__", "unknown"))


In [ ]:
# Edit these for your current run.
CONFIG_PATH = Path("~/codex/mIF-pipeline/prototyping/prototype_v2-Crop.yaml").expanduser()
SLIDE_ID = "SLIDE-0329_crop_2048"

# Stage execution toggles.
FORCE = False
RUN_SETUP = True
RUN_MERGE = True
RUN_NIMBUS = True
RUN_ASSEMBLE = True

# If a stage toggle is False, the notebook still shows the call pattern via dry-run.
SETUP_DRY_RUN = not RUN_SETUP
MERGE_DRY_RUN = not RUN_MERGE
NIMBUS_DRY_RUN = not RUN_NIMBUS
ASSEMBLE_DRY_RUN = not RUN_ASSEMBLE


In [ ]:
def show(value):
    print(json.dumps(value, indent=2, default=str))


def stage_call(label, func, *args, **kwargs):
    print(f"\n=== {label} ===")
    try:
        result = func(*args, **kwargs)
        show(result)
        return result
    except Exception as exc:
        print(f"{type(exc).__name__}: {exc}")
        traceback.print_exc()
        return None


In [ ]:
print("config path:", CONFIG_PATH)
print("config exists:", CONFIG_PATH.exists())

config = load_config(CONFIG_PATH)
slide = get_slide_config(config, SLIDE_ID)

print("slide_id:", SLIDE_ID)
print("slide_dir:", slide["slide_dir"])
print("output_dir:", slide["output_dir"])
print("pixel_size_um:", slide["pixel_size_um"])

show({
    "full_merge": slide.get("full_merge"),
    "instanseg": slide.get("instanseg"),
    "mask_export": slide.get("mask_export"),
    "nimbus": slide.get("nimbus"),
    "spatialdata": slide.get("spatialdata"),
})


In [ ]:
artifact_paths = spatialdata_builder_module._spatialdata_paths(slide)
show({key: str(value) for key, value in artifact_paths.items()})

print("Expected InstanSeg merged image input:", slide["full_merge"]["ome_path"])
print("Expected InstanSeg channel subset:", slide["instanseg"]["channels"])
print("Expected final SpatialData image input:", artifact_paths["full_merge_path"])
print("Expected whole-cell labels:", artifact_paths["cell_mask_path"])
print("Expected nuclear labels:", artifact_paths["nuclear_mask_path"])
print("Expected optional Nimbus table:", artifact_paths["nimbus_table_path"])
print("Expected final SpatialData store:", artifact_paths["store_path"])


## Setup / Channel Map Generation

This stage generates or validates the channel map from the configured `slide_dir` and setup patterns. It is the intended first step when you want to test alias generation before merge.


In [ ]:
setup_result = stage_call(
    f"setup_slide({SLIDE_ID}, dry_run={SETUP_DRY_RUN})",
    setup_slide,
    config,
    SLIDE_ID,
    force=FORCE,
    dry_run=SETUP_DRY_RUN,
)


In [ ]:
channel_map_path = Path(slide["channel_map_file"])
print("channel_map_file:", channel_map_path)
print("channel map exists:", channel_map_path.exists())

if channel_map_path.exists():
    channel_map = json.loads(channel_map_path.read_text())
    print("channel count:", len(channel_map))
    show(channel_map[: min(10, len(channel_map))])
else:
    print("Run setup with RUN_SETUP = True to write the generated channel map to disk.")


## Merge Stage

This stage is safe to run in the merge/InstanSeg environment. It writes both merge artifacts and establishes the segmentation input file path.


In [ ]:
merge_result = stage_call(
    f"merge_slide_ometiffs({SLIDE_ID}, dry_run={MERGE_DRY_RUN})",
    merge_slide_ometiffs,
    config,
    SLIDE_ID,
    force=FORCE,
    dry_run=MERGE_DRY_RUN,
)


## Optional Nimbus Stage

Nimbus is optional now. If it is disabled in config, the final SpatialData assembly should still succeed with image + labels only.


In [ ]:
nimbus_enabled = bool((slide.get("nimbus") or {}).get("enabled", False))
print("Nimbus enabled:", nimbus_enabled)

if nimbus_enabled:
    nimbus_result = stage_call(
        f"run_nimbus_chunked({SLIDE_ID}, dry_run={NIMBUS_DRY_RUN})",
        run_nimbus_chunked,
        config,
        SLIDE_ID,
        force=FORCE,
        dry_run=NIMBUS_DRY_RUN,
    )
else:
    nimbus_result = None
    print("Nimbus is disabled in this config.")


## Final SpatialData Assembly

Run this stage in the modern Harpy + SpatialData environment. It reads the existing file artifacts and writes the canonical final SpatialData store.


In [ ]:
assemble_result = stage_call(
    f"assemble_spatialdata({SLIDE_ID}, dry_run={ASSEMBLE_DRY_RUN})",
    assemble_spatialdata,
    config,
    SLIDE_ID,
    force=FORCE,
    dry_run=ASSEMBLE_DRY_RUN,
    return_sdata=not ASSEMBLE_DRY_RUN,
)


In [ ]:
if assemble_result and not ASSEMBLE_DRY_RUN and "sdata" in assemble_result:
    sdata = assemble_result["sdata"]
    print("images:", list(sdata.images.keys()))
    print("labels:", list(sdata.labels.keys()))
    print("shapes:", list(sdata.shapes.keys()))
    print("tables:", list(sdata.tables.keys()))
    print("aggregate tables:")
    for table_summary in assemble_result.get("aggregate_tables", []):
        show(table_summary)
else:
    print("No in-memory SpatialData object available. Set RUN_ASSEMBLE = True in a modern Harpy + SpatialData env to inspect the assembled object.")
